# College-going rate by high school (2020-21)

- **Paper:** (prep)
- **Original notebook:** `2_by_school_type.ipynb`
- **Reads:**
  - `data/raw/California_State_Geoportal/.../cgr12mo21.txt`
  - `data/raw/CA_school_directory/CDESchoolDirectoryExport.txt`
  - `data/raw/SVI/...`
  - `data/raw/geographic_boundaries/ca_tracts_2022`
- **Writes:**
  - `outputs/enrollment_by_school.pkl`
  - `outputs/svi_enrollment_by_type.pkl`
  - `outputs/hs_types.pkl`

> Note: The save of enrollment_by_school.pkl was commented out in the original. It is turned back on here.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pickle

# set directory

In [ ]:
# data import

enrollment = pd.read_csv(f"{DATA_DIR}/school_related/20-21/cgr12mo21.txt",sep='\t')
# https://www.cde.ca.gov/SchoolDirectory/ExportSelect?simpleSearch=N&address=&city=&counties=&districts=&cdscode=&charter=&magnet=&name=&nps=&search=0&zip=&yearround=&status=1%2C2&types=&order=1&qsc=13956&qdc=13956
schools = pd.read_csv(f"{DATA_DIR}/CA_school_directory/CDESchoolDirectoryExport.txt",sep = '\t')

tracts = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/ca_tracts_2022')

svi_20 = gpd.read_file(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI2020_CALIFORNIA_tract.gdb')

svi_16 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2016_CA.csv',dtype = object)
svi_18 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2018_CA.csv',dtype = object)



In [ ]:
# cleaning

# getting schoolwide completers and enrollment data
schoolwide = enrollment[enrollment['AggregateLevel'] == 'S']
schoolwide = schoolwide[schoolwide['ReportingCategory'] == 'TA']
schoolwide = schoolwide[schoolwide['CompleterType']=='TA'] # assuming we want all high school completers regardless of A-G rqmts/ graduates
schoolwide= schoolwide.replace('*', np.nan)

# joining schoolwide enrollment data to school directory info
# pub_schools = schools[schools['Public Yes/No'] == 'Y']
schools = schools.rename(columns={'School': 'SchoolName'})

schools = schools.rename(columns={'District': 'DistrictName'})

schools = pd.merge(schoolwide,schools, on = ['SchoolName','DistrictName'],how = 'inner')
schools = schools[schools['Entity Type']!= 'Elementary Schools (Public)']
# schools = schools[schools['Entity Type']=='High Schools (Public)']
with open(f'{OUTPUT_DIR}/enrollment_by_school.pkl','wb') as f:
  pickle.dump(schools,f)


hs_types = schools['Entity Type'].unique()
hs_types
types = list()

for n, hs in enumerate(hs_types):
 types.append(schools[schools['Entity Type'] == hs])


def locate_schools(df):
   # df.loc[:, 'Latitude'].astype(float)
    #df.loc[:, 'Longitude'].astype(float)
    located_schools = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs='EPSG:4326')
    return located_schools
# end function
located_schools = list()
for i in range(len(types)):
  located_schools.append(locate_schools(types[i]))
# end for

# schools within tracts


tracts = tracts.to_crs(4326)
schools_in = list()

for i in range(len(located_schools)):
  schools_in.append(gpd.sjoin(located_schools[i], tracts, predicate='intersects'))
  # concat FIPS column

  schools_in[i]['FIPS'] = schools_in[i]['STATEFP'] + schools_in[i]['COUNTYFP']+schools_in[i]['TRACTCE']
# end for



In [ ]:
#merge SVIs
svi_16 = svi_16[['FIPS','RPL_THEMES']]
svi_18 = svi_18[['FIPS','RPL_THEMES']]
svi_20 = svi_20[['FIPS','RPL_THEMES']]

svi_16 = svi_16.rename(columns = {'RPL_THEMES':'SVI_2016'})
svi_18 = svi_18.rename(columns = {'RPL_THEMES':'SVI_2018'})
svi_20 = svi_20.rename(columns = {'RPL_THEMES':'SVI_2020'})
svi_20 = svi_20.merge(svi_16,on = ["FIPS"],suffixes = ('_20','_16'))
svi_20 = svi_20.merge(svi_18,on = ["FIPS"],suffixes = ('20','_18'))




svi_20['SVI_2016'] = svi_20['SVI_2016'].astype(float)
svi_20['SVI_2018'] = svi_20['SVI_2018'].astype(float)
svi_20['SVI_2020'] = svi_20['SVI_2020'].astype(float)
svi_20[svi_20 == -999] = np.nan




In [ ]:
# analysis
#average SVI 2016-2020

svi_20['RPL_THEMES'] = svi_20[['SVI_2016', 'SVI_2018', 'SVI_2020']].mean(axis=1)


# enrollment rate per tract public
completers = [0]*len(schools_in)
college = [0]*len(schools_in)
enrolled = [0]*len(schools_in)
svi_enrolled = [0]*len(schools_in)
correlation = [0]*len(schools_in)

for i in range(len(schools_in)):
  schools_in[i]['High School Completers'] = schools_in[i]['High School Completers'].astype(float)
  schools_in[i]['Enrolled In College - Total (12 Months)'] = schools_in[i]['Enrolled In College - Total (12 Months)'].astype(float)
  completers[i] = schools_in[i].groupby('FIPS')['High School Completers'].sum()
  college[i] = schools_in[i].groupby('FIPS')['Enrolled In College - Total (12 Months)'].sum()
  enrolled[i] = pd.merge(completers[i],college[i],on = 'FIPS')
  enrolled[i]['enrollment_rate'] = enrolled[i]['Enrolled In College - Total (12 Months)']/enrolled[i]['High School Completers']
  svi_enrolled[i] = pd.merge(svi_20,enrolled[i],on = 'FIPS')
  correlation[i] = svi_enrolled[i]['RPL_THEMES'].corr(svi_enrolled[i]['enrollment_rate'])
# end for

# corr

svi_enrolled_pub = svi_enrolled[5]
svi_enrolled_pub
correlation_pub = svi_enrolled_pub['RPL_THEMES'].corr(svi_enrolled_pub['enrollment_rate'])
print('correlation (public):',correlation_pub)



In [ ]:
# plotting


fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 15))

# Iterate over each group of svi_enrolled data and plot on respective subplot
for i, ax in enumerate(axes.flat):
    if i < 11:  # Ensure not to exceed the number of plots
        ax.scatter(svi_enrolled[i]['RPL_THEMES'], svi_enrolled[i]['enrollment_rate'])
        ax.set_title('College Enrollment Rate vs. SVI\nfor ' + hs_types[i])
        ax.set_xlabel('Social Vulnerability Index')
        ax.set_ylabel('College Enrollment Rate')
        ax.grid(False)  # Add grid lines for better visualization
    else:
        ax.axis('off')  # Turn off empty subplots
    # end if/else
# end for
plt.tight_layout()
plt.show()


In [ ]:
svi_enrolled_pub
plt.scatter(svi_enrolled_pub['RPL_THEMES'],svi_enrolled_pub['enrollment_rate'])
plt.title('College Enrollment Rate vs. SVI\nfor ' + hs_types[5])
plt.xlabel('Social Vulnerability Index')
plt.ylabel('College Enrollment Rate')

In [ ]:
# categorizing by quartiles for SVI
# https://www.atsdr.cdc.gov/placeandhealth/svi/faq_svi.html

bins = [0, 0.25, 0.5, 0.75, 1]
labels = ["Q1", "Q2", "Q3", "Q4"]

fig, axes = plt.subplots(4, 3, figsize=(15, 15))

for i in range(len(svi_enrolled)):
    svi_enrolled[i]['SVI_quartile'] = pd.cut(svi_enrolled[i]['RPL_THEMES'], bins=bins, labels=labels, include_lowest=True)

    grouped_data = [svi_enrolled[i][svi_enrolled[i]['SVI_quartile'] == label]['enrollment_rate'] for label in labels]

    row = i % 4
    col = i // 4

    axes[row, col].hist(grouped_data, bins=20, label=labels, stacked=True, edgecolor='white')
    axes[row, col].set_xlabel('Enrollment Rate')
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].set_title('Enrollment in SVI Quartiles\nfor ' + hs_types[i])
    axes[row, col].legend()
    axes[row, col].grid(False)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming svi_enrolled is a list of DataFrames for different types of high schools
bins = [0, 0.25, 0.5, 0.75, 1]
labels = ["Q1", "Q2", "Q3", "Q4"]

fig, ax = plt.subplots()  # Create a single plot

# Assuming svi_enrolled[5] refers to the DataFrame for the 6th type of high school
df = svi_enrolled[5].copy()  # Make a copy to avoid modifying the original DataFrame
df['SVI_quartile'] = pd.cut(df['RPL_THEMES'], bins=bins, labels=labels, include_lowest=True)

grouped_data = [df[df['SVI_quartile'] == label]['enrollment_rate'] for label in labels]

ax.hist(grouped_data, bins=20, label=labels, stacked=True, edgecolor='white')

ax.set_xlabel('Enrollment Rate')
ax.set_ylabel('Frequency')
ax.set_title('Enrollment in SVI Quartiles\nfor ' + hs_types[5])  # Assuming hs_types contains labels for each type of high school
ax.legend()
ax.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
with open(f'{OUTPUT_DIR}/svi_enrollment_by_type.pkl','wb') as f:
  pickle.dump(svi_enrolled,f)

with open(f'{OUTPUT_DIR}/hs_types.pkl','wb') as f:
  pickle.dump(hs_types,f)